# Ablation A1 — Không có Categorical Features

**Mục đích**: Đánh giá đóng góp của Entity Embedding bằng cách loại bỏ hoàn toàn 4 static categorical features (`Store ID`, `Product ID`, `Category`, `Region`).

| Variant | Categorical | Model |
|---------|-------------|-------|
| **A1 (this)** | ❌ Không có | LSTM (numerical only) |
| A4 (Proposed) | ✅ Entity Embedding | LSTM + Entity Embedding |

**Horizons**: 7, 14, 28 ngày  
**Output**: `result/ablation_A1_no_cat_summary.csv` + `result/ablation_A1_no_cat_details.csv`

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from sklearn.preprocessing import StandardScaler
import warnings, os
import kagglehub

warnings.filterwarnings('ignore')
tf.random.set_seed(42)
np.random.seed(42)


# Download latest version
DATA_DIR  = kagglehub.dataset_download("atomicd/retail-store-inventory-and-demand-forecasting")
DATA_PATH = os.path.join(DATA_DIR, "sales_data.csv")
RESULT_DIR = "/kaggle/working/"

TARGET    = 'Units Sold'
TRAIN_END = '2023-06-30'
VAL_END   = '2023-10-31'
HORIZONS  = [7, 14, 28]
LOOKBACK  = 30
LAG       = 7
print("Path to dataset files:", DATA_PATH)
# ── Ablation tag ─────────────────────────────────────────────────────────────
ABLATION_NAME = 'A1-NoCategorical'
print(f'Ablation: {ABLATION_NAME}')

## 1. Load & Feature Engineering (không có categorical)

In [ ]:
df_raw = pd.read_csv(DATA_PATH)
df_raw['Date'] = pd.to_datetime(df_raw['Date'])
df_raw = df_raw.sort_values(['Store ID', 'Product ID', 'Date']).reset_index(drop=True)

# ── Time-varying categoricals → ordinal encode vào numerical sequence ────────
WEATHER_MAP = {'Sunny': 0, 'Cloudy': 1, 'Rainy': 2, 'Snowy': 3, 'Windy': 4, 'Stormy': 5}
SEASON_MAP  = {'Winter': 0, 'Spring': 1, 'Summer': 2, 'Fall': 3}
df_raw['weather_enc'] = df_raw['Weather Condition'].map(WEATHER_MAP).fillna(0).astype(int)
df_raw['season_enc']  = df_raw['Seasonality'].map(SEASON_MAP).fillna(0).astype(int)

# ── Numerical features (giống proposed, nhưng KHÔNG có static cat) ───────────
grp = df_raw.groupby(['Store ID', 'Product ID'])[TARGET]
df_raw['lag_7']           = grp.shift(7)
df_raw['lag_14']          = grp.shift(14)
df_raw['lag_28']          = grp.shift(28)
df_raw['rolling_mean_7']  = grp.transform(lambda x: x.shift(1).rolling(7).mean())
df_raw['rolling_mean_14'] = grp.transform(lambda x: x.shift(1).rolling(14).mean())
df_raw['day_of_week']     = df_raw['Date'].dt.dayofweek
df_raw['day_of_month']    = df_raw['Date'].dt.day
df_raw['month']           = df_raw['Date'].dt.month
df_raw['is_weekend']      = (df_raw['day_of_week'] >= 5).astype(int)
df_raw = df_raw.bfill().fillna(0)

NUM_COLS = [
    TARGET,
    'Price', 'Discount', 'Competitor Pricing',
    'Inventory Level', 'Units Ordered',
    'Promotion', 'Epidemic',
    'weather_enc', 'season_enc',
    'lag_7', 'lag_14', 'lag_28',
    'rolling_mean_7', 'rolling_mean_14',
    'day_of_week', 'day_of_month', 'month', 'is_weekend',
]
TARGET_IDX  = NUM_COLS.index(TARGET)
series_keys = sorted(df_raw.groupby(['Store ID', 'Product ID']).groups.keys())

print(f'Series: {len(series_keys)} | Num features: {len(NUM_COLS)} | Static cat: 0 (ablated)')

## 2. Model — Numerical Only (không có Entity Embedding)

In [ ]:
def build_model_no_cat(lookback, n_num, horizon, lstm_units=64, dropout=0.2):
    """
    Giống proposed nhưng KHÔNG có entity embedding.
    Input duy nhất: num_input (batch, lookback, n_num)
    """
    num_input = layers.Input(shape=(lookback, n_num), name='num_input')
    x = layers.LSTM(lstm_units, return_sequences=True)(num_input)
    x = layers.Dropout(dropout)(x)
    x = layers.LSTM(lstm_units // 2)(x)
    x = layers.Dropout(dropout)(x)
    out = layers.Dense(horizon)(x)

    model = models.Model(inputs=num_input, outputs=out)
    model.compile(optimizer='adam', loss='mse')
    return model

demo = build_model_no_cat(LOOKBACK, len(NUM_COLS), horizon=7)
demo.summary()

## 3. Build Dataset

In [ ]:
def make_sequences_no_cat(num_arr, target_idx, lookback, horizon, stride=7):
    X_num, y = [], []
    for i in range(lookback, len(num_arr) - horizon + 1, stride):
        X_num.append(num_arr[i - lookback:i])
        y.append(num_arr[i:i + horizon, target_idx])
    return np.array(X_num, dtype=np.float32), np.array(y, dtype=np.float32)


def build_global_arrays_no_cat(horizon, lookback):
    X_num_tr, X_num_vl = [], []
    y_tr, y_vl = [], []
    scalers = {}

    for store, product in series_keys:
        sdf = df_raw[(df_raw['Store ID'] == store) & (df_raw['Product ID'] == product)]
        sdf = sdf.set_index('Date')
        key = f'{store}_{product}'

        train_num = sdf[:TRAIN_END][NUM_COLS].values.astype(np.float32)
        val_num   = sdf[:VAL_END][NUM_COLS].values.astype(np.float32)

        scaler = StandardScaler().fit(train_num)
        scalers[key] = scaler

        tr_scaled  = scaler.transform(train_num)
        val_scaled = scaler.transform(val_num)

        Xn_tr, yt = make_sequences_no_cat(tr_scaled,  TARGET_IDX, lookback, horizon)
        Xn_vl, yv = make_sequences_no_cat(val_scaled, TARGET_IDX, lookback, horizon)

        X_num_tr.append(Xn_tr); X_num_vl.append(Xn_vl)
        y_tr.append(yt);        y_vl.append(yv)

    return (
        np.concatenate(X_num_tr), np.concatenate(y_tr),
        np.concatenate(X_num_vl), np.concatenate(y_vl),
        scalers
    )

print('Dataset builder ready.')

## 4. Rolling Evaluation

In [ ]:
def rolling_eval_no_cat(model, scaler, store, product, horizon, lookback):
    sdf = df_raw[(df_raw['Store ID'] == store) & (df_raw['Product ID'] == product)].set_index('Date')

    full_scaled = scaler.transform(sdf[NUM_COLS].values.astype(np.float32))
    eval_start  = pd.Timestamp(VAL_END) + pd.Timedelta(days=1)
    eval_end    = sdf.index.max()

    all_fc, all_ac = [], []
    t = eval_start
    while t + pd.Timedelta(days=horizon - 1) <= eval_end:
        t_loc     = sdf.index.get_loc(t)
        win_start = t_loc - lookback
        if win_start < 0:
            t += pd.Timedelta(days=horizon); continue
        actual = sdf[TARGET][t: t + pd.Timedelta(days=horizon - 1)].values
        if len(actual) < horizon:
            t += pd.Timedelta(days=horizon); continue

        X_num = full_scaled[win_start:t_loc][np.newaxis]  # (1, lookback, n_num)

        fc_scaled = model.predict(X_num, verbose=0)[0]  # (horizon,)
        dummy = np.zeros((horizon, len(NUM_COLS)), dtype=np.float32)
        dummy[:, TARGET_IDX] = fc_scaled
        fc = np.clip(scaler.inverse_transform(dummy)[:, TARGET_IDX], 0, None)

        all_fc.append(fc)
        all_ac.append(actual.astype(np.float32))
        t += pd.Timedelta(days=horizon)

    if not all_fc:
        return {k: np.nan for k in ['smape', 'mase', 'rmse', 'rmsle']}

    fc_arr, ac_arr = np.array(all_fc), np.array(all_ac)
    train_vals = sdf[TARGET][:TRAIN_END].values.astype(np.float32)
    lag   = min(LAG, len(train_vals) - 1)
    denom = np.mean(np.abs(train_vals[lag:] - train_vals[:-lag])) or 1.0

    return {
        'smape': (2 * np.abs(fc_arr - ac_arr) / (np.abs(fc_arr) + np.abs(ac_arr) + 1e-8)).mean() * 100,
        'mase' : np.mean(np.abs(fc_arr - ac_arr)) / denom,
        'rmse' : float(np.sqrt(np.mean((fc_arr - ac_arr) ** 2))),
        'rmsle': float(np.sqrt(np.mean((np.log1p(np.clip(fc_arr, 0, None)) - np.log1p(np.clip(ac_arr, 0, None))) ** 2))),
    }

print('Evaluation function ready.')

## 5. Train & Evaluate

In [ ]:
os.makedirs(RESULT_DIR, exist_ok=True)

summary_rows = []
detail_rows  = []

for h in HORIZONS:
    print(f'\n=== Horizon = {h} ===')
    print('  Building arrays...')
    X_num_tr, y_tr, X_num_vl, y_vl, scalers = build_global_arrays_no_cat(h, LOOKBACK)
    print(f'  Train: {X_num_tr.shape} | Val: {X_num_vl.shape}')

    model = build_model_no_cat(LOOKBACK, len(NUM_COLS), horizon=h)

    cb = callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
    model.fit(
        X_num_tr, y_tr,
        validation_data=(X_num_vl, y_vl),
        epochs=50, batch_size=256,
        callbacks=[cb], verbose=0
    )

    print('  Rolling eval on TEST...')
    scores = {k: [] for k in ['smape', 'mase', 'rmse', 'rmsle']}
    for store, product in series_keys:
        key = f'{store}_{product}'
        r   = rolling_eval_no_cat(model, scalers[key], store, product, h, LOOKBACK)
        for k in scores: scores[k].append(r[k])
        print(f"    {store} | {product} | sMAPE={r['smape']:.2f}% MASE={r['mase']:.4f} RMSE={r['rmse']:.2f} RMSLE={r['rmsle']:.4f}")
        detail_rows.append({
            'ablation': ABLATION_NAME,
            'model': 'LSTM-NoCategorical',
            'store': store, 'product': product,
            'horizon': h, 'lookback': LOOKBACK,
            'smape': round(float(r['smape']), 4),
            'mase':  round(float(r['mase']),  4),
            'rmse':  round(float(r['rmse']),  4),
            'rmsle': round(float(r['rmsle']), 4),
        })

    row = {
        'ablation':      ABLATION_NAME,
        'model':         'LSTM-NoCategorical',
        'dataset':       'retail_inventory_daily',
        'target':        TARGET,
        'horizon':       h,
        'lookback':      LOOKBACK,
        'mean_smape':    round(float(np.nanmean(scores['smape'])), 4),
        'median_smape':  round(float(np.nanmedian(scores['smape'])), 4),
        'mean_mase':     round(float(np.nanmean(scores['mase'])), 4),
        'median_mase':   round(float(np.nanmedian(scores['mase'])), 4),
        'mean_rmse':     round(float(np.nanmean(scores['rmse'])), 4),
        'median_rmse':   round(float(np.nanmedian(scores['rmse'])), 4),
        'mean_rmsle':    round(float(np.nanmean(scores['rmsle'])), 4),
        'median_rmsle':  round(float(np.nanmedian(scores['rmsle'])), 4),
    }
    summary_rows.append(row)
    print(f"  H={h} | sMAPE={row['mean_smape']:.2f}% MASE={row['mean_mase']:.4f} RMSE={row['mean_rmse']:.2f} RMSLE={row['mean_rmsle']:.4f}")

# ── Save ─────────────────────────────────────────────────────────────────────
summary_path = f'{RESULT_DIR}/ablation_A1_no_cat_summary.csv'
detail_path  = f'{RESULT_DIR}/ablation_A1_no_cat_details.csv'
pd.DataFrame(summary_rows).to_csv(summary_path, index=False)
pd.DataFrame(detail_rows).to_csv(detail_path,   index=False)
print(f'\nSaved summary → {summary_path}')
print(f'Saved details → {detail_path}')
pd.DataFrame(summary_rows)

## 6. So sánh với Proposed (A4 — Entity Embedding)

In [ ]:
import glob

# Load proposed result (best tuned — GAO)
proposed_files = glob.glob(f'{RESULT_DIR}/*gao*summary*') + glob.glob(f'{RESULT_DIR}/*entity_emb_summary*')
if proposed_files:
    df_proposed = pd.read_csv(proposed_files[0])
    df_proposed['ablation'] = 'A4-EntityEmbedding (Proposed)'
else:
    print('Proposed result not found — skipping comparison.')
    df_proposed = None

df_a1 = pd.DataFrame(summary_rows)

if df_proposed is not None:
    compare_cols = ['ablation', 'horizon', 'mean_smape', 'mean_mase', 'mean_rmse', 'mean_rmsle']
    df_compare = pd.concat([
        df_a1[compare_cols],
        df_proposed[compare_cols]
    ], ignore_index=True).sort_values(['horizon', 'ablation'])
    print('\n=== Ablation Comparison ===')
    print(df_compare.to_string(index=False))
else:
    print(df_a1[['ablation', 'horizon', 'mean_smape', 'mean_mase', 'mean_rmse', 'mean_rmsle']].to_string(index=False))